
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Lab - Model Monitoring
In this notebook, you will monitor the performance of a deployed machine learning model using Databricks. You will enable an inference table, send batched requests, and set up comprehensive monitoring.

**Lab Outline:**

_In this lab, you will need to complete the following tasks:_
- **Task 1:** Save the Training Data as Reference for Drift
- **Task 2:** Processing and Monitoring Inference Data
> - **2.1:** Monitoring the Inference Table  
> - **2.2:** Processing Inference Table Data 
> - **2.3:** Analyzing Processed Requests
- **Task 3:** Persisting Processed Model Logs
- **Task 4:** Setting Up and Monitoring Inference Data
> - **4.1:** Creating an Inference Monitor with Databricks Lakehouse Monitoring
> - **4.2:** Inspect and Monitor Metrics Tables


📝 **Your task:** Complete the **`<FILL_IN>`** sections in the code blocks and follow the other steps as instructed.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.
Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.
1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
   - In the drop-down, select **More**.
   - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.
  
**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:
1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.
1. Find the triangle icon to the right of your compute cluster name and click it.
1. Wait a few minutes for the cluster to start.
1. Once the cluster is running, complete the steps above to select your cluster.

## Requirements
Please review the following requirements before starting the lesson:
- To run this notebook, you need to use one of the following Databricks runtime(s): **`16.3.x-cpu-ml-scala2.12`**

## Classroom Setup

Before starting the Lab, run the provided classroom setup script. This script will define configuration variables necessary for the Lab. Execute the following cell:

🚨 **_Please wait for the classroom setup to run, as it may take around 10 minutes to execute and create a model that you will be using for the Lab._**

> ****🚨Note:**** If you encounter a "file not found" error when running this cell, simply re-run it. This issue may occur as we transition from DBFS to Unity Catalog.

In [0]:
%run ../Includes/Classroom-Setup-3.Lab

# 3.0 - Model Preparation
In this notebook, you will prepare a machine learning model using a banking dataset. You will train a classification model and deploy it using Databricks.

**Steps:**
- **Step 1:** Prepare Model
> - **Step 1.1:** Load Dataset
> - **Step 1.2:** Train / New Requests Split
> - **Step 1.3:** Fit a Classification Model
- **Step 2:** Deploying the Model

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Inference table created successfully.


## Step 1: Prepare Model

### Step 1.1: Load Dataset
- Load the CSV file into a Spark DataFrame.
- Rename columns to remove invalid characters.
- Convert to Pandas DataFrame and display it.

...

2025/10/07 01:01:34 INFO mlflow.tracking._tracking_service.client: 🏃 View run burly-midge-999 at: dbc-bcd7adfe-8f71.cloud.databricks.com/ml/experiments/1142554600511480/runs/cfc58db822904c2f8fcb2a4bb47c1fa1.
2025/10/07 01:01:34 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: dbc-bcd7adfe-8f71.cloud.databricks.com/ml/experiments/1142554600511480.


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

Registered model 'dbacademy.labuser11256191_1759794214.loan_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

Created version '2' of model 'dbacademy.labuser11256191_1759794214.loan_model'.


**Other Conventions:**

Throughout this lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

Username:          labuser11256191_1759794214@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser11256191_1759794214
Working Directory: /Volumes/dbacademy/ops/labuser11256191_1759794214@vocareum_com
Dataset Location:  NestedNamespace (banking='/Volumes/dbacademy_banking/v01', cdc_diabetes='/Volumes/dbacademy_cdc_diabetes/v01', monitoring='/Volumes/dbacademy_monitoring/v01', telco='/Volumes/dbacademy_telco/v01')


###Load Data
Load the banking dataset into a Pandas DataFrame and prepare it for training and requesting.

In [0]:
import pandas as pd
from pyspark.sql.functions import lit, col
from pyspark.sql.types import DoubleType

# Load the Delta table into a Spark DataFrame
dataset_path = loan_pd_df


###Train / New Requests Split
Split the data into training and request sets.



In [0]:
# Split the data into train and request sets
train_df, request_df = loan_df.randomSplit(weights=[0.6, 0.4], seed=42)

# Convert to Pandas DataFrames
train_pd_df = train_df.toPandas()
request_pd_df = request_df.toPandas()
target_col = "Personal_Loan"
ID = "ID"

X_request = request_df.drop(target_col)
y_request = request_df.select(target_col)

###Define Model Name and Display the Model Serving Endpoint
- Set the model name for registration in the Databricks Model Registry.

- Display the Model Serving Endpoint URL for easy access.

In [0]:
model_name = f"{DA.catalog_name}.{DA.schema_name}.loan_model"

## Task 1: Save the Training Data as Reference for Drift
 
Save the training data as a Delta table to serve as a reference for detecting data drift.

**Steps:**
1. **Convert Data:** Convert the Pandas DataFrame to a Spark DataFrame.
2. **Save Data:** Save the Spark DataFrame as a Delta table.
3. **Read and Update Data:** Read the Delta table and update the data types to match the required schema.


In [0]:
from pyspark.sql.functions import lit, col
from pyspark.sql.types import DoubleType
from pyspark.sql import DataFrame, functions as F, types as T

## Convert Pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(train_pd_df).withColumn('model_id', lit(0)).withColumn("labeled_data", col("Personal_Loan").cast(DoubleType()))

(spark_df
  .write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", True)
  .option("delta.enableChangeDataFeed", "true")
  .saveAsTable(f"{DA.catalog_name}.{DA.schema_name}.lab_baseline_features"))

## Read the existing table into a DataFrame
baseline_features_df = spark.table(f"{DA.catalog_name}.{DA.schema_name}.lab_baseline_features")

## Cast the labeled_data and CCAvg columns to INT
baseline_features_df = baseline_features_df.withColumn('labeled_data', F.col('labeled_data').cast(T.IntegerType()))
baseline_features_df = baseline_features_df.withColumn('CCAvg', F.col('CCAvg').cast(T.IntegerType()))

## Overwrite the existing table with the updated DataFrame
(baseline_features_df
  .write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{DA.catalog_name}.{DA.schema_name}.lab_baseline_features"))

## Task 2: Monitoring and Processing Inference Data 
This task involves processing the logged data, monitoring the inference table, and analyzing the processed requests to ensure continuous availability and accuracy of model performance data.




### Task 2.1: Monitoring the Inference Table
In this task, you will monitor the inference table to ensure that data is populating correctly as the model starts receiving requests.

**Steps:**
  1. **Monitor Table:** Monitor the inference table to ensure data is populating correctly.
  2. **Check Table:** Implement a loop to wait and check for the table population, retrying until the data appears.


In [0]:
## Attempt to read the table
inference_df = spark.read.table(f"{DA.catalog_name}.{DA.schema_name}.lab_model_inference_table")

## Check if the table is not empty
if inference_df.count() > 0:
    ## If successful and the table is not empty, display the DataFrame and break the loop
    display(inference_df)
else:
    ## If the table is empty, print table in empty
    print("Table is empty, trying again in 10 seconds.")

client_request_id,databricks_request_id,date,timestamp_ms,status_code,execution_time_ms,request,response,sampling_fraction,request_metadata
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,1727532116521,200,7,"{""dataframe_split"": {""index"": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], ""columns"": [""index"", ""ID"", ""Age"", ""Experience"", ""Income"", ""ZIP_Code"", ""Family"", ""CCAvg"", ""Education"", ""Mortgage"", ""Securities_Account"", ""CD_Account"", ""Online"", ""CreditCard""], ""data"": [[0, 1, 25, 1, 49, 91107, 4, 1.6, 1, 0, 1, 0, 0, 0], [1, 3, 39, 15, 11, 94720, 1, 1.0, 1, 0, 0, 0, 0, 0], [2, 5, 35, 8, 45, 91330, 4, 1.0, 2, 0, 0, 0, 0, 1], [3, 7, 53, 27, 72, 91711, 2, 1.5, 2, 0, 0, 0, 1, 0], [4, 9, 35, 10, 81, 90089, 3, 0.6, 2, 104, 0, 0, 1, 0], [5, 10, 34, 9, 180, 93023, 1, 8.9, 3, 0, 0, 0, 0, 0], [6, 12, 29, 5, 45, 90277, 3, 0.1, 2, 0, 0, 0, 1, 0], [7, 14, 59, 32, 40, 94920, 4, 2.5, 2, 0, 0, 0, 1, 0], [8, 15, 67, 41, 112, 91741, 1, 2.0, 1, 0, 1, 0, 0, 0], [9, 16, 60, 30, 22, 95054, 1, 1.5, 3, 0, 0, 0, 1, 1]]}}","List(List(0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0))",1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)"
null,2c241a15-b4da-44b2-b126-4f9575132337,2024-09-28,1727532117030,200,7,"{""dataframe_split"": {""index"": [30, 31, 32, 33, 34, 35, 36, 37, 38, 39], ""columns"": [""index"", ""ID"", ""Age"", ""Experience"", ""Income"", ""ZIP_Code"", ""Family"", ""CCAvg"", ""Education"", ""Mortgage"", ""Securities_Account"", ""CD_Account"", ""Online"", ""CreditCard""], ""data"": [[30, 57, 55, 30, 29, 94005, 3, 0.1, 2, 0, 1, 1, 1, 0], [31, 62, 47, 21, 125, 93407, 1, 5.7, 1, 112, 1, 0, 0, 0], [32, 63, 42, 18, 22, 90089, 1, 1.0, 1, 0, 0, 0, 0, 0], [33, 65, 47, 23, 105, 90024, 2, 3.3, 1, 0, 0, 0, 0, 0], [34, 70, 53, 29, 20, 90045, 4, 0.2, 1, 0, 0, 0, 1, 0], [35, 71, 42, 18, 115, 91335, 1, 3.5, 1, 0, 0, 0, 0, 1], [36, 75, 28, 3, 135, 94611, 2, 3.3, 1, 0, 0, 0, 0, 1], [37, 76, 31, 7, 135, 94901, 4, 3.8, 2, 0, 0, 1, 1, 1], [38, 78, 46, 20, 29, 92220, 3, 0.5, 2, 0, 0, 0, 0, 0], [39, 82, 47, 22, 40, 94612, 3, 2.7, 2, 0, 0, 0, 1, 0]]}}","List(List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0))",1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)"
null,62bad5ca-5337-4752-8a79-267ea0fc2db4,2024-09-28,1727532117486,200,6,"{""dataframe_split"": {""index"": [60, 61, 62, 63, 64, 65, 66, 67, 68, 69], ""columns"": [""index"", ""ID"", ""Age"", ""Experience"", ""Income"", ""ZIP_Code"", ""Family"", ""CCAvg"", ""Education"", ""Mortgage"", ""Securities_Account"", ""CD_Account"", ""Online"", ""CreditCard""], ""data"": [[60, 146, 59, 35, 124, 90007, 1, 7.4, 1, 0, 0, 0, 0, 1], [61, 147, 46, 19, 84, 94102, 1, 2.67, 2, 0, 0, 0, 1, 1], [62, 150, 48, 22, 42, 93955, 3, 2.2, 2, 0, 0, 0, 0, 0], [63, 151, 46, 22, 118, 94107, 2, 7.5, 1, 0, 0, 1, 1, 1], [64, 153, 57, 32, 24, 93117, 1, 1.3, 1, 0, 0, 0, 1, 1], [65, 160, 61, 35, 41, 94545, 4, 1.7, 2, 0, 1, 0, 1, 0], [66, 161, 29, 0, 134, 95819, 4, 6.5, 3, 0, 0, 0, 0, 0], [67, 165, 53, 27, 92, 95120, 2, 1.1, 1, 0, 1, 0, 0, 0], [68, 169, 50, 26, 13, 91320, 4, 1.0, 1, 0, 0, 0, 1, 0], [69, 170, 27, 1, 112, 90503, 4, 2.1, 3, 0, 0, 0, 0, 1]]}}","List(List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0))",1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)"
null,c5be40f6-4814-4d99-a0a1-9f8d9c72bc4a,2024-09-28,1727532117517,200,6,"{""dataframe_split"": {""index"": [70, 71, 72, 73, 74, 75, 76, 77, 78, 79], ""columns"": [""index"", ""ID"", ""Age"", ""Experience"", ""Income"", ""ZIP_Code"", ""Family"", ""CCAvg"", ""Education"", ""Mortgage"", ""Securities_Account"", ""CD_Account"", ""Online"", ""CreditCard""], ""data"": [[70, 172, 52, 28, 11, 95817, 3, 0.4, 1, 0, 1, 0, 0,

### Task 2.2: Processing Inference Table Data
In this task, you will extract and analyze the data logged in the inference table to prepare it for monitoring.

**Steps:**
  1. **Define Functions:** Define helper functions for JSON conversion.
  2. **Process Requests:** Process the raw requests and unpack JSON payloads.
  3. **Convert Data:** Convert timestamps and explode batched requests into individual rows.


In [0]:
from pyspark.sql import DataFrame, functions as F, types as T
import json
import pandas as pd

"""
Conversion helper functions.
"""
def convert_to_record_json(json_str: str) -> str:
    """
    Converts records from the four accepted JSON formats for Databricks
    Model Serving endpoints into a common, record-oriented
    DataFrame format which can be parsed by the PySpark function from_json.
    
    :param json_str: The JSON string containing the request or response payload.
    :return: A JSON string containing the converted payload in record-oriented format.
    """
    try:
        request = json.loads(json_str)
    except json.JSONDecodeError:
        return json_str
    output = []
    if isinstance(request, dict):
        obj_keys = set(request.keys())
        if "dataframe_records" in obj_keys:
            # Record-oriented DataFrame
            output.extend(request["dataframe_records"])
        elif "dataframe_split" in obj_keys:
            # Split-oriented DataFrame
            dataframe_split = request["dataframe_split"]
            output.extend([dict(zip(dataframe_split["columns"], values)) for values in dataframe_split["data"]])
        elif "instances" in obj_keys:
            # TF serving instances
            output.extend(request["instances"])
        elif "inputs" in obj_keys:
            # TF serving inputs
            output.extend([dict(zip(request["inputs"], values)) for values in zip(*request["inputs"].values())])
        elif "predictions" in obj_keys:
            # Predictions
            output.extend([{'predictions': prediction} for prediction in request["predictions"]])
        return json.dumps(output)
    else:
        # Unsupported format, pass through
        return json_str


@F.pandas_udf(T.StringType())
def json_consolidation_udf(json_strs: pd.Series) -> pd.Series:
    """A UDF to apply the JSON conversion function to every request/response."""
    return json_strs.apply(convert_to_record_json)

In [0]:
from pyspark.sql import DataFrame, functions as F, types as T
from pyspark.sql.types import TimestampType 

def process_requests(requests_raw: DataFrame) -> DataFrame:
    """
    Processes a stream of raw requests and:
        - Unpacks JSON payloads for requests
        - Extracts relevant features as scalar values (first element of each array)
        - Converts Unix epoch millisecond timestamps to Spark TimestampType
    """
    ## Calculate the current timestamp in seconds
    current_ts = int(spark.sql("SELECT unix_timestamp(current_timestamp())").collect()[0][0])

    ## Define the start timestamp for 30 days ago
    start_ts = current_ts - 30 * 24 * 60 * 60  # 30 days in seconds

    ## Dynamically calculate the min and max values of timestamp_ms
    min_max = requests_raw.agg(
        F.min("timestamp_ms").alias("min_ts"),
        F.max("timestamp_ms").alias("max_ts")
    ).collect()[0]

    min_ts = min_max["min_ts"] / 1000  # Convert from milliseconds to seconds
    max_ts = min_max["max_ts"] / 1000  # Convert from milliseconds to seconds

    ## Transform timestamp_ms to span the last month
    requests_timestamped = requests_raw.withColumn(
        'timestamp', 
        (start_ts + ((F.col("timestamp_ms") / 1000 - min_ts) / (max_ts - min_ts)) * (current_ts - start_ts)).cast(TimestampType())
    ).drop("timestamp_ms")

    ## Consolidate and unpack JSON.
    requests_unpacked = requests_timestamped \
        .withColumn("request", json_consolidation_udf(F.col("request"))) \
        .withColumn('request', F.from_json(F.col("request"), F.schema_of_json('[{"ID": 1.0,"Age": 40.0, "Experience": 10.0, "Income": 84.0, "ZIP_Code": 9302.0, "Family": 3.0, "CCAvg": 2.0, "Education": 2.0, "Mortgage": 0.0, "Securities_Account": 0.0, "CD_Account": 0.0, "Online": 1.0, "CreditCard": 1.0}]'))) \
        .withColumn("response", F.expr("transform(response.predictions, x -> x)"))  \
        .withColumn('response', F.col("response"))

    ## Explode batched requests into individual rows.
    DB_PREFIX = "__db"
    requests_exploded = requests_unpacked \
        .withColumn(f"{DB_PREFIX}_request_response", F.arrays_zip(F.col("request"), F.col("response"))) \
        .withColumn(f"{DB_PREFIX}_request_response", F.explode(F.col(f"{DB_PREFIX}_request_response"))) \
        .select(F.col("*"), F.col(f"{DB_PREFIX}_request_response.request.*"), F.col(f"{DB_PREFIX}_request_response.response").alias("Personal_Loan")) \
        .drop(f"{DB_PREFIX}_request_response", "request", "response") \
        .withColumn('model_id', F.lit(0))

    return requests_exploded

### Task 2.3: Analyzing Processed Requests

After processing and unpacking the logged data from the inference table, the next step is to analyze the requests that were successfully answered by the model, filtering and joining with additional label data for comprehensive analysis.

**Steps:**
  1. **Process Data:** Filter and analyze the requests that were successfully answered by the model.
  2. **Join Data:** Join with additional label data for comprehensive analysis.


In [0]:
# Process the inference data
model_logs_df = process_requests(inference_df.where("status_code = 200")) # Let's ignore bad requests

# Ensure the ID column is added during processing
display(model_logs_df)

client_request_id,databricks_request_id,date,status_code,execution_time_ms,sampling_fraction,request_metadata,timestamp,Age,CCAvg,CD_Account,CreditCard,Education,Experience,Family,ID,Income,Mortgage,Online,Securities_Account,ZIP_Code,Personal_Loan,model_id
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,25.0,1.6,0.0,0.0,1.0,1.0,4.0,1.0,49.0,0.0,0.0,1.0,91107.0,0.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,39.0,1.0,0.0,0.0,1.0,15.0,1.0,3.0,11.0,0.0,0.0,0.0,94720.0,0.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,35.0,1.0,0.0,1.0,2.0,8.0,4.0,5.0,45.0,0.0,0.0,0.0,91330.0,0.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,53.0,1.5,0.0,0.0,2.0,27.0,2.0,7.0,72.0,0.0,1.0,0.0,91711.0,0.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,35.0,0.6,0.0,0.0,2.0,10.0,3.0,9.0,81.0,104.0,1.0,0.0,90089.0,0.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,34.0,8.9,0.0,0.0,3.0,9.0,1.0,10.0,180.0,0.0,0.0,0.0,93023.0,1.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,29.0,0.1,0.0,0.0,2.0,5.0,3.0,12.0,45.0,0.0,1.0,0.0,90277.0,0.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,59.0,2.5,0.0,0.0,2.0,32.0,4.0,14.0,40.0,0.0,1.0,0.0,94920.0,0.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,67.0,2.0,0.0,0.0,1.0,41.0,1.0,15.0,112.0,0.0,0.0,1.0,91741.0,0.0,0
null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,60.0,1.5,0.0,1.0,3.0,30.0,1.0,16.0,22.0,0.0,1.0,0.0,95054.0,0.0,0


In [0]:
# Convert Pandas DataFrame to Spark DataFrame
loan_spark_df = spark.createDataFrame(loan_pd_df)

# Rename 'Personal_Loan' to 'labeled_data' in the loan_spark_df
label_spark_df = loan_spark_df.withColumnRenamed('Personal_Loan', 'labeled_data')

# Join with model_logs_df
model_logs_df_labeled = model_logs_df.join(
    label_spark_df.select("ID", "labeled_data"), 
    on="ID", 
    how="left"
)

# Display the joined DataFrame
display(model_logs_df_labeled)

ID,client_request_id,databricks_request_id,date,status_code,execution_time_ms,sampling_fraction,request_metadata,timestamp,Age,CCAvg,CD_Account,CreditCard,Education,Experience,Family,Income,Mortgage,Online,Securities_Account,ZIP_Code,Personal_Loan,model_id,labeled_data
1.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,25.0,1.6,0.0,0.0,1.0,1.0,4.0,49.0,0.0,0.0,1.0,91107.0,0.0,0,0
3.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,39.0,1.0,0.0,0.0,1.0,15.0,1.0,11.0,0.0,0.0,0.0,94720.0,0.0,0,0
5.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,35.0,1.0,0.0,1.0,2.0,8.0,4.0,45.0,0.0,0.0,0.0,91330.0,0.0,0,0
7.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,53.0,1.5,0.0,0.0,2.0,27.0,2.0,72.0,0.0,1.0,0.0,91711.0,0.0,0,0
9.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,35.0,0.6,0.0,0.0,2.0,10.0,3.0,81.0,104.0,1.0,0.0,90089.0,0.0,0,0
10.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,34.0,8.9,0.0,0.0,3.0,9.0,1.0,180.0,0.0,0.0,0.0,93023.0,1.0,0,1
12.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,29.0,0.1,0.0,0.0,2.0,5.0,3.0,45.0,0.0,1.0,0.0,90277.0,0.0,0,0
14.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,59.0,2.5,0.0,0.0,2.0,32.0,4.0,40.0,0.0,1.0,0.0,94920.0,0.0,0,0
15.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,67.0,2.0,0.0,0.0,1.0,41.0,1.0,112.0,0.0,0.0,1.0,91741.0,0.0,0,0
16.0,null,bc466050-63fb-4dd8-bf48-e2bdd6051c4b,2024-09-28,200,7,1.0,"Map(model_name -> lab_cddcfabfaa_model_monitor.ws_3551974319838082.loan_model, endpoint_name -> anupriya-malik-databricks-com_MLas04_M03_Lab, model_version -> 1)",2025-09-07T01:01:49Z,60.0,1.5,0.0,1.0,3.0,30.0,1.0,22.0,0.0,1.0,0.0,95054.0,0.0,0,0


## Task 3: Persisting Processed Model Logs

In this task, you will save the enriched model logs to ensure long-term availability for ongoing monitoring and analysis.

**Steps:**
1. **Convert Data Types:** Convert all columns to appropriate data types.
2. **Save Logs:** Save the processed logs to a designated storage for long-term use.
3. **Enable CDF:** Enable Change Data Feed (CDF) to facilitate efficient incremental processing of new data.


In [0]:
from pyspark.sql import functions as F, types as T

## Convert all columns to IntegerType except those of type MAP<STRING, STRING>
for col_name, col_type in model_logs_df_labeled.dtypes:
    if col_type != 'map<string,string>':
        model_logs_df_labeled = model_logs_df_labeled.withColumn(
            col_name, 
            F.col(col_name).cast(T.IntegerType())
        )

## Overwrite the existing table with the updated DataFrame
model_logs_df_labeled.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DA.catalog_name}.{DA.schema_name}.lab_model_logs")

For efficient execution, enable CDF (Change Data Feed) so monitoring can incrementally process the data.

In [0]:
spark.sql(f'ALTER TABLE {DA.catalog_name}.{DA.schema_name}.lab_model_logs SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')

DataFrame[]

## Task 4: Setting Up and Monitoring Inference Data

This task includes setting up the monitoring of inference data and ensuring its continuous availability for analysis and monitoring.



### Task 4.1: Creating an Inference Monitor with Databricks Lakehouse Monitoring
Set up monitoring to continuously track model performance and detect any anomalies or drift in real-time.
**Steps:**
  1. **Configure Monitor:** Configure and initiate the monitoring of model logs.
  2. **Create Monitor:** Create an inference monitor and validate its creation.
  3. **Verify Metrics:** Verify that metrics tables are created and populated.


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorInferenceLog, MonitorInferenceLogProblemType, MonitorInfoStatus, MonitorRefreshInfoState, MonitorMetric

w = WorkspaceClient()
table_name = f'{DA.catalog_name}.{DA.schema_name}.lab_model_logs'
baseline_table_name = f"{DA.catalog_name}.{DA.schema_name}.lab_baseline_features"

## ML problem type, either "classification" or "regression"
PROBLEM_TYPE = MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION

## Window sizes to analyze data over
GRANULARITIES = ["5 minutes"]

## Directory to store generated dashboard
ASSETS_DIR = f"/Workspace/Users/{DA.username}/databricks_lakehouse_monitoring/lab_model_logs"

## Optional parameters
SLICING_EXPRS = ["age<20", "age>70", "CreditCard='1'", "Income<20", "Income>120"]   # Expressions to slice data with
print(f"Creating monitor for lab_model_logs")

info = w.quality_monitors.create(
  table_name=table_name,
  inference_log=MonitorInferenceLog(
    timestamp_col='timestamp',
    granularities=GRANULARITIES,
    model_id_col='model_id', # Model version number 
    prediction_col='Personal_Loan',
    problem_type=PROBLEM_TYPE,
    label_col='labeled_data' # Optional
  ),
  baseline_table_name=baseline_table_name,
  slicing_exprs=SLICING_EXPRS,
  output_schema_name=f"{DA.catalog_name}.{DA.schema_name}",
  assets_dir=ASSETS_DIR
)

Creating monitor for lab_model_logs


#### Instructions for Accessing Your Monitor Dashboard

Follow these steps to ensure that your quality monitor is correctly set up and to access the monitor dashboard:

1. **Wait for Monitor Creation**: Start by initiating a loop to check the monitor’s status. The loop will query the monitor’s status every 10 seconds until it changes from *pending* to *active*, confirming that the monitor has been successfully created.

2. **Check for Metric Refreshes**: Once the monitor is created, it automatically triggers a metric refresh. Ensure that this refresh process is completed successfully.

3. **Monitor Refresh Status**: Identify the first refresh operation and enter a loop to check its state. If the state is either *pending* or *running*, wait for 30 seconds before checking again. This loop will continue until the refresh state changes to *success*, confirming the refresh operation has completed successfully.

4. **Access the Monitor Dashboard**: Once the monitor is active and metrics have been refreshed, a URL to the monitor dashboard will be constructed using the workspace URL, catalog name, schema name, and specifying the `quality` tab. This URL is printed out, allowing you direct access to the dashboard where you can review quality metrics for your model logs.

By following these steps, you can ensure your quality monitor is set up correctly and use the provided URL to conveniently access the dashboard to monitor your model's performance and data quality.


In [0]:
import time

## Wait for monitor to be created
while info.status == MonitorInfoStatus.MONITOR_STATUS_PENDING:
  info = w.quality_monitors.get(table_name=table_name)
  time.sleep(10)

assert info.status == MonitorInfoStatus.MONITOR_STATUS_ACTIVE, "Error creating monitor"
## A metric refresh will automatically be triggered on creation
refreshes = w.quality_monitors.list_refreshes(table_name=table_name).refreshes
assert(len(refreshes) > 0)

run_info = refreshes[0]
while run_info.state in (MonitorRefreshInfoState.PENDING, MonitorRefreshInfoState.RUNNING):
  run_info = w.quality_monitors.get_refresh(table_name=table_name, refresh_id=run_info.refresh_id)
  time.sleep(30)

assert run_info.state == MonitorRefreshInfoState.SUCCESS, "Monitor refresh failed"

w.quality_monitors.get(table_name=table_name)
## Extract workspace URL
workspace_url = spark.conf.get('spark.databricks.workspaceUrl')

## Construct the monitor dashboard URL
monitor_dashboard_url = f"https://{workspace_url}/explore/data/{DA.catalog_name}/{DA.schema_name}/lab_model_logs?o={DA.schema_name}&activeTab=quality"

print(f"Monitor Dashboard URL: {monitor_dashboard_url}")

### Task 4.2: Inspect and Monitor Metrics Tables

In this task, you will learn how to inspect and monitor the metrics tables generated by the Databricks quality monitoring tools. These tables provide valuable insights into the performance and behavior of your models, including summary statistics and data drift detection.

You will perform the following:
- **Inspect the Metrics Tables Using UI**: Locate and review the profile and drift metrics tables created by the monitoring process. These tables are saved in your default database and provide detailed metrics and visualizations.

**Inspect the Metrics Tables Using UI**


By default, the metrics tables are saved in the default database.

The `create_monitor` call created two new tables: the profile metrics table and the drift metrics table.

- **Profile Metrics Table**: This table records summary statistics for each column in the monitored table.
- **Drift Metrics Table**: This table records metrics that compare current values in the monitored table to baseline values, identifying potential drift.

These tables use the same name as the primary table to be monitored, with the suffixes `_profile_metrics` and `_drift_metrics`.

> ****Instructions:****
> 1. Go to the Table where the monitor is created: `(Table name=f'{DA.catalog_name}.{DA.schema_name}.model_logs')`.
> 2. Check the output tables:
>    - Locate the table with the suffix `_profile_metrics` to view summary statistics for each column.
>    - Locate the table with the suffix `_drift_metrics` to view metrics that compare current values to baseline values.
> 3. View the dashboard associated with these metrics tables.
> 4. Explore the different metrics and visualizations created. The dashboard provides insights into data distribution, potential data drift, and other key metrics.


**⚠️ Note: You may see a CAST_INVALID_INPUT error in the Numerical Feature Quantile Drift widget. This occurs because the automatically generated query attempts to cast the "Baseline" string to a timestamp in the Window column. This doesn’t affect other metrics or drift detection functionality and can be safely ignored for this Lab.**

# Explore the dashboard!

# Conclusion
In this lab, you successfully deployed a machine learning model and set up a monitoring framework to track its performance. You sent batched requests to the model endpoint and monitored the responses to detect any anomalies or drift. Additionally, you explored how to use Databricks Lakehouse Monitoring to continuously track and alert on model performance metrics.


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>